# Classical vs QAOA Benchmark Tutorial

This notebook demonstrates a structured comparison of:

1. Pure classical BM baseline  
2. Standard QAOA  
3. Preconditioned QAOA  
4. Warm-started preconditioning  
5. Global BM step on QAOA-preconditioned graph  

The goal is to transparently evaluate whether QAOA improves over strong classical baselines.

In [11]:
from pathlib import Path
from dataclasses import replace
import sys
from dataclasses import replace
from ModularQaoaSetup.qaoa_solvers.modular import ModularSolveConfig


cwd = Path.cwd()

if cwd.name == "notebooks" and cwd.parent.name == "ModularQaoaSetup":
    repo_root = cwd.parent.parent
elif cwd.name == "notebooks":
    repo_root = cwd.parent
else:
    repo_root = cwd

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from ModularQaoaSetup import (
    assemble_portable_solver,
    load_weighted_graph,
)
from ModularQaoaSetup.benchmarking import benchmark_modular_run
from ModularQaoaSetup.benchmarking_classical import (
    sweep_seeds_best,
    polish_like_preconditioning,
    solver_bm_fast,
)
from ModularQaoaSetup.qaoa_solvers.modular import ModularSolveConfig

data_dir = repo_root / "data"
parquet_files = list(data_dir.glob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"No parquet files found in {data_dir}")

graph_path = max(parquet_files, key=lambda p: p.stat().st_size)
G = load_weighted_graph(graph_path)

print("Graph:", graph_path.name)
print("Nodes:", G.number_of_nodes(), "Edges:", G.number_of_edges())

Graph: dataset_b.parquet
Nodes: 180 Edges: 226


## 1️⃣ Classical BM Baseline

We first establish a strong classical reference using a seed sweep. Then we use a polishing step to further improve the best solution found. This sets a high bar for QAOA to beat.

In [7]:

best_score, best_seed, best_assignment = sweep_seeds_best(
    G,
    solver=solver_bm_fast,
    seeds=range(5000),
)

print("Best BM score:", best_score)
print("Best seed:", best_seed)

Best BM score: 7097.270255421455
Best seed: 3682


### Optional polishing (BM + greedy)

This mirrors the post-processing used inside preconditioning.

In [13]:

# Minimal config just to carry BM parameters
cfg_polish = ModularSolveConfig(
    partition_strategy="multilevel",
    region_optimizer="hybrid_preconditioned",
    boundary_optimizer="hybrid_preconditioned",
    depth=1,
    max_block_size=16,
    rounds=1,
    restarts=1,
    maxiter=20,
    seed=best_seed,
)

polished_score, polished_assignment = polish_like_preconditioning(
    G,
    best_assignment,
    cfg_polish,
)

print("Best BM score:", best_score)
print("After polish:", polished_score)

Best BM score: 7097.270255421455
After polish: 7099.571726101163


## 2️⃣ Standard QAOA (no preconditioning)

In [19]:
cfg_std_qaoa = ModularSolveConfig(
    partition_strategy="multilevel",
    region_optimizer="standard_qaoa",
    boundary_optimizer="standard_qaoa",
    depth=1,          # p=1 baseline
    max_block_size=18,
    rounds=2,
    restarts=2,
    maxiter=25,
    seed=3682,
    error_model="ideal",
)

res_qaoa = benchmark_modular_run(G, name=graph_path.name, config=cfg_std_qaoa, separate_timing_pass=False)

print("Standard QAOA cut:", res_qaoa.weighted_cut)

Standard QAOA cut: 6900.8833970081505


## 3️⃣ Preconditioned QAOA

In [21]:
cfg_pre = ModularSolveConfig(
    partition_strategy="multilevel",
    region_optimizer="hybrid_preconditioned",
    boundary_optimizer="hybrid_preconditioned",
    depth=1,
    max_block_size=16,
    rounds=2,
    restarts=2,
    maxiter=25,
    seed=3682,
    max_light_cone_size=16,
    preconditioning_backend="burer_monteiro",
    preconditioner_shots=0,
    preconditioner_use_pcut=True,
)

res_pre = benchmark_modular_run(G, name=graph_path.name, config=cfg_pre)

print("Preconditioned QAOA cut:", res_pre.weighted_cut)

Preconditioned QAOA cut: 6996.071600050634


## 4️⃣ Warm-Start Preconditioning

Inject the best classical assignment into the solver.

In [22]:
from dataclasses import replace

cfg_warm = replace(
    cfg_pre,
    warm_start_assignment=best_assignment,
)

res_warm = benchmark_modular_run(G, name=graph_path.name, config=cfg_warm)

print("Warm-start cut:", res_warm.weighted_cut)

Warm-start cut: 7099.571726101163


## 5️⃣ Global BM Step on QAOA-Preconditioned Graph

This isolates the effect of QAOA-induced graph deformation.

In [23]:
cfg_global = replace(
    cfg_pre,
    warm_start_assignment=best_assignment,
    preconditioning_mode="global_bm_step",
)

res_global = benchmark_modular_run(G, name=graph_path.name, config=cfg_global)

print("Global BM step cut:", res_global.weighted_cut)

Global BM step cut: 7099.571726101163


## Summary

Compare:

- Classical BM baseline  
- Polished classical  
- Standard QAOA  
- Preconditioned QAOA  
- Warm-started preconditioning  
- Global BM step  

This provides a transparent evaluation of whether QAOA improves upon strong classical references.